# Инференс: запрос → рекомендации

Пользователь (история прослушиваний + профиль) пишет запрос на русском, в ответ — текст и список треков.
Вся логика в пакете `recsys/`, здесь только запуск.

* `experiments.ipynb` — почему такая выдача: разбор по шагам, сравнение вариантов, LLM
* `evaluate.py` — метрики на валидации
* `examples/candgen.ipynb` — проверка сервисов BM25 и HNSW руками

## 1. Настройки

По умолчанию — настоящий датасет и оба сервиса на сервере:

* `USE_CANDGEN = True` — BM25 и HNSW с сервера по HTTP (`configs/candgen.yaml`). Адреса и ключи — `BM25_URL`, `BM25_API_KEY`,
  `HNSW_URL`, `HNSW_API_KEY` (файл `.env` в корне или секреты проекта DataSphere). `False` — всё считается локально, без HNSW.
* `DATA = "crs"` — датасет из `DATA_DIR` (файлы целиком или частями `train-00000-of-00082.parquet`);
  `"synthetic"` — сгенерированные данные, работает без файлов (только чтобы проверить код).

In [22]:
USE_CANDGEN = True         # BM25 и HNSW с сервера; False — всё локально, без HNSW
DATA = "crs"               # настоящий датасет; "synthetic" — сгенерированные примеры, только чтобы проверить код
DATA_DIR = "CRS dataset"   # tracks_meta, train, test_public (целиком или частями)
N_USERS = 300              # сколько пользователей загрузить (None — все, долго)

# адреса и ключи сервисов — из файла .env или секретов DataSphere; можно и здесь:
# import os; os.environ["BM25_URL"] = "http://<IP>:8000"; os.environ["BM25_API_KEY"] = "..."
# os.environ["HNSW_URL"] = "http://<IP>:8001"; os.environ["HNSW_API_KEY"] = "..."

In [ ]:
%pip install -q -r requirements.txt

## 2. Каталог и пайплайн

Загружается один раз. Для сервисов печатается, доступны ли они и какая версия индекса.

In [23]:
%load_ext autoreload
%autoreload 2

from dataclasses import replace

from recsys.config import load_config
from recsys.data import load_data
from recsys.pipeline import Pipeline
from recsys.schemas import Message

files = ["configs/default.yaml"] + (["configs/candgen.yaml"] if USE_CANDGEN else [])
cfg = load_config(files, {"data": {"source": DATA, "crs": {"dir": DATA_DIR, "n_users": N_USERS}}})
data = load_data(cfg)
pipe = Pipeline.from_config(cfg, data.catalog)

for source in pipe.retrievers:
    if getattr(source, "remote", False):
        print(source.name, source.health())

[data] crs: DataBundle(tracks=64016, requests: train=1412, test_public=1344) за 2.8 c


ValueError: Неизвестный тип источника кандидатов: hnsw (type=hnsw_api)

## 3. Запрос

Пользователь из `test_public`: его история, профиль и настоящий запрос; известен трек, который он на самом деле
послушал (цель). `REQUEST` — номер запроса; `TEXT` — свой текст вместо запроса из датасета (история та же).

In [ ]:
REQUEST = 0      # номер запроса в test_public (загружено N_USERS пользователей, до 5 запросов у каждого)
TEXT = None      # None — настоящий запрос из датасета (известен трек, который послушали); или свой текст:
# TEXT = "хочу что-нибудь спокойное под вечер, без рэпа"

user = data.requests[REQUEST]
# запрос нужного типа: user = next(r for r in data.requests if r.meta["query_family"] == "exact")
request = replace(user, dialog=[Message("user", TEXT)]) if TEXT else user
print("запрос:", request.dialog[-1].text)
if not TEXT:
    print("тип:", request.meta["query_family"], "| цель:", request.meta["artist"], "—", request.meta["title"],
          "| уже слушал:", not request.meta["is_new"])
print("треков в истории:", len(request.history))
print("профиль:", request.user_info[:200], "...")

## 4. Ответ

In [ ]:
response = pipe.run(request, debug=True)
print(response.text)
response.to_frame()

Если запрос из датасета: на каком месте трек, который пользователь послушал.

In [ ]:
if not TEXT:
    target = request.target_ids[0]
    place = response.track_ids.index(target) + 1 if target in response.track_ids else "нет в top-10"
    print("цель:", request.meta["artist"], "—", request.meta["title"], "| место в выдаче:", place)

## 5. Почему такая выдача

Как понят запрос (английские теги — именно они уходят в BM25) и сколько кандидатов дал каждый источник.
`relisten` — треки из своей истории, `bm25_*` и `hnsw` — сервисы на сервере, остальные считаются здесь.

In [ ]:
print("понял как:", response.summary.text)
print("теги:", response.summary.include_tags, "| исключить:", response.summary.exclude_tags)
print("кандидатов:", response.debug["n_candidates"])

## 6. Следующая реплика

Диалог продолжается: уже показанное не повторяется, новые пожелания перекрывают старые.

In [ ]:
request2 = request.next_turn(response, "а теперь поэнергичнее")
response2 = pipe.run(request2)
print(response2.text)
response2.to_frame()